In [2]:
import json
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

def search_articles(query, k=10):
    """
    구글 SERP API를 통해 뉴스/문서 검색을 수행합니다.
    """
    wrapper = GoogleSerperAPIWrapper(k=k)
    return wrapper.results(query)

def summarize_articles(docs, query):
    """
    검색된 문서들을 요약하여 주요 내용을 정리합니다.
    """
    pages = " ".join([d.page_content for d in docs])
    llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
    template = """{docs} \n {query} \n 요약해줘"""
    chain = LLMChain(llm=llm, prompt=PromptTemplate(input_variables=["docs", "query"], template=template))
    return chain.run(docs=pages, query=query)

def process_search(query):
    """
    검색 → 문서 로딩 → 텍스트 분할의 전체 파이프라인
    """
    response = search_articles(query)
    urls = [hit['link'] for hit in response['organic'][:3]]  # best guess
    loader = UnstructuredURLLoader(urls=urls)
    docs = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    return text_splitter.split_documents(docs)